In [8]:
"""
Loan Default Prediction with Keras (LendingClub Dataset)
--------------------------------------------------------

This notebook builds a deep neural network (DNN) using TensorFlow/Keras
on LendingClub loan data to predict whether a loan will be repaid or charged off.

Covers:
- Data cleaning and preprocessing
- Feature engineering
- Train/test split
- Feature scaling
- DNN model building and training
- Evaluation with classification report and confusion matrix
"""


'\nLoan Default Prediction with Keras (LendingClub Dataset)\n--------------------------------------------------------\n\nThis notebook builds a deep neural network (DNN) using TensorFlow/Keras\non LendingClub loan data to predict whether a loan will be repaid or charged off.\n\nCovers:\n- Data cleaning and preprocessing\n- Feature engineering\n- Train/test split\n- Feature scaling\n- DNN model building and training\n- Evaluation with classification report and confusion matrix\n'

In [9]:
# Basic Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# TensorFlow and Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Sklearn Utilities
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import confusion_matrix, classification_report


In [10]:

DATA_PATH = '/mnt/mls/data/udemy/lending_club_loan_two.csv'

df = pd.read_csv(DATA_PATH)


In [11]:

# Create new target
loan_status_to_repaid = {
    'Fully Paid': 1,
    'Charged Off': 0
}
df['loan_repaid'] = df['loan_status'].map(loan_status_to_repaid)

# Drop loan_status now
df.drop('loan_status', axis=1, inplace=True)


In [12]:

# Drop irrelevant or high-missingness columns
emp_title_cols = ['emp_title', 'title']
df.drop(columns=emp_title_cols, inplace=True)

# Fill mort_acc based on correlation with total_acc
mort_acc_mean = df.groupby('total_acc')['mort_acc'].mean()

def fill_mort_acc(total_acc, mort_acc):
    if np.isnan(mort_acc):
        return mort_acc_mean[total_acc]
    else:
        return mort_acc

df['mort_acc'] = df.apply(lambda x: fill_mort_acc(x['total_acc'], x['mort_acc']), axis=1)

# Drop remaining small missing percentages
df.dropna(inplace=True)


In [13]:
def clean_term(term):
    if isinstance(term, str) and term.strip():
        try:
            return int(term.split(' ')[0])
        except ValueError:
            return np.nan
    return np.nan

df['term'] = df['term'].apply(clean_term)


# Drop grade, keep sub_grade
df.drop('grade', axis=1, inplace=True)

# Dummies for sub_grade
subgrade_dummies = pd.get_dummies(df['sub_grade'], drop_first=True)
df = pd.concat([df.drop('sub_grade', axis=1), subgrade_dummies], axis=1)

# Dummies for categorical features
cat_feats = ['verification_status', 'application_type', 'initial_list_status', 'purpose', 'home_ownership', 'zip_code']

# Handle home ownership
df['home_ownership'] = df['home_ownership'].replace(['NONE', 'ANY'], 'OTHER')

# Extract zip_code
df['zip_code'] = df['address'].apply(lambda x: x[-5:])

# Drop address
df.drop('address', axis=1, inplace=True)

# Dummies
df = pd.get_dummies(df, columns=cat_feats, drop_first=True)

# Handle earliest_cr_line
df['earliest_cr_year'] = df['earliest_cr_line'].apply(lambda date: int(date[-4:]))
df.drop('earliest_cr_line', axis=1, inplace=True)

# Drop issue_d (data leakage)
df.drop('issue_d', axis=1, inplace=True)


In [14]:

X = df.drop('loan_repaid', axis=1).values
y = df['loan_repaid'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=101)


In [16]:
print(df.select_dtypes(include='object').columns)


Index(['emp_length'], dtype='object')


In [17]:
df['emp_length']

0         10+ years
1           4 years
2          < 1 year
3           6 years
4           9 years
            ...    
396025      2 years
396026      5 years
396027    10+ years
396028    10+ years
396029    10+ years
Name: emp_length, Length: 376929, dtype: object

In [15]:

scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


ValueError: could not convert string to float: '9 years'

In [ ]:

model = Sequential([
    Dense(78, activation='relu'),
    Dense(39, activation='relu'),
    Dense(19, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy')


In [ ]:

history = model.fit(
    X_train, y_train,
    epochs=25,
    batch_size=256,
    validation_data=(X_test, y_test)
)


In [ ]:

losses = pd.DataFrame(model.history.history)
losses.plot()
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.show()


In [ ]:

y_pred = (model.predict(X_test) > 0.5).astype(int)

print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


In [ ]:

import random
random.seed(101)

random_ind = random.randint(0, len(df))
new_customer = df.drop('loan_repaid', axis=1).iloc[random_ind]
actual_label = df['loan_repaid'].iloc[random_ind]

prediction = (model.predict(new_customer.values.reshape(1, -1)) > 0.5).astype(int)

print(f"Model prediction: {prediction[0][0]}")
print(f"Actual repayment status: {actual_label}")
